# CEOAI Practice 1 - Star Observatory Minimum Solution

Objective: produce the first valid 600-row-style submission:

1. Estimate star centers with an intensity-weighted centroid.
2. Train a simple flux regressor from image summary features.
3. Export center and flux rows in the required CSV format.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

In [2]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
print(train.head())
print({"train": len(train), "test": len(test)})

    image_id  fried_parameter   airmass  target_flux
0  00000.png         1.320525  1.328511  3104.153567
1  00001.png         0.953145  1.279548  2574.724655
2  00002.png         0.661622  1.309472  2044.014880
3  00003.png         0.914006  1.790308  1971.258685
4  00004.png         1.539929  1.648373  3052.443416
{'train': 80, 'test': 24}


In [3]:
def load_gray(path):
    return np.asarray(Image.open(path).convert("L"), dtype=float)

def image_features(image):
    yy, xx = np.mgrid[0:image.shape[0], 0:image.shape[1]]
    weights = np.maximum(image - np.percentile(image, 98), 0)
    total = weights.sum() + 1e-9
    cx = float((weights * xx).sum() / total)
    cy = float((weights * yy).sum() / total)
    return {
        "sum": float(image.sum()),
        "mean": float(image.mean()),
        "std": float(image.std()),
        "max": float(image.max()),
        "centroid_x": cx,
        "centroid_y": cy,
    }

def build_feature_frame(df, folder):
    rows = []
    for image_id in df["image_id"]:
        image = load_gray(DATA / folder / image_id)
        rows.append({"image_id": image_id, **image_features(image)})
    return pd.DataFrame(rows)

train_features = build_feature_frame(train, "train_images").merge(train, on="image_id")
test_features = build_feature_frame(test, "test_images")
train_features.head()

,image_id,sum,mean,std,max,centroid_x,centroid_y,fried_parameter,airmass,target_flux
0,00000.png,44548.0,2.718994,2.363901,37.0,43.022327,55.529412,1.320525,1.328511,3104.153567
1,00001.png,45442.0,2.773560,2.359193,33.0,38.111964,34.206321,0.953145,1.279548,2574.724655
2,00002.png,46007.0,2.808044,2.328771,27.0,83.388479,56.070409,0.661622,1.309472,2044.014880
3,00003.png,45149.0,2.755676,2.122370,25.0,58.058140,98.041279,0.914006,1.790308,1971.258685
4,00004.png,44827.0,2.736023,2.348866,38.0,37.165296,77.021636,1.539929,1.648373,3052.443416


In [4]:
feature_cols = ["sum", "mean", "std", "max", "centroid_x", "centroid_y"]
X_train, X_val, y_train, y_val = train_test_split(
    train_features[feature_cols],
    np.log1p(train_features["target_flux"]),
    test_size=0.25,
    random_state=0,
)
flux_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
flux_model.fit(X_train, y_train)
val_pred = np.expm1(flux_model.predict(X_val))
print({"fixture_flux_rmse": float(np.sqrt(mean_squared_error(np.expm1(y_val), val_pred)))})

{'fixture_flux_rmse': 136.4543962841185}


In [5]:
test_flux = np.expm1(flux_model.predict(test_features[feature_cols]))
rows = []
for i, row in test_features.iterrows():
    image_id = row["image_id"]
    rows.append({
        "subtaskID": 1,
        "datapointID": image_id,
        "answer": f"({row['centroid_x']:.2f}, {row['centroid_y']:.2f})",
    })
    rows.append({
        "subtaskID": 2,
        "datapointID": image_id,
        "answer": float(test_flux[i]),
    })

submission = pd.DataFrame(rows)
submission.to_csv(OUT / "submission.csv", index=False)
assert len(submission) == 2 * len(test)
submission.head()

,subtaskID,datapointID,answer
0,1,00000.png,"(63.90, 83.90)"
1,2,00000.png,2696.679466
2,1,00001.png,"(28.14, 81.65)"
3,2,00001.png,2559.520513
4,1,00002.png,"(80.30, 42.31)"


In [6]:
hidden_path = DATA / "test_hidden.csv"
if hidden_path.exists():
    hidden = pd.read_csv(hidden_path).merge(test_features, on="image_id")
    center_mae = mean_absolute_error(hidden[["center_x", "center_y"]], hidden[["centroid_x", "centroid_y"]])
    flux_rmse = np.sqrt(mean_squared_error(hidden["target_flux"], test_flux))
    print({"fixture_center_coord_mae": float(center_mae), "fixture_flux_rmse": float(flux_rmse)})

print("wrote", OUT / "submission.csv")

{'fixture_center_coord_mae': 0.5641946522288774, 'fixture_flux_rmse': 212.83801692129322}
wrote d:\projects\Supervised-Learning-Experiments\olympiads\competition_samples\raw\ceoai-2026-practice-rounds\round-1\star_observatory\outputs\submission.csv
